In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.5 Shape tracing and tests

Attention bugs are almost always shape bugs or mask bugs. Trace the shapes for a
batched, multi-head input, then lock the invariant that makes a decoder valid:
no token sees the future.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
batch, n_head, tokens, hs = 2, 4, 6, 8
q = torch.randn(batch, n_head, tokens, hs)
k = torch.randn(batch, n_head, tokens, hs)
v = torch.randn(batch, n_head, tokens, hs)
out, w = scaled_dot_product_attention(q, k, v, causal=True)
print("in:", (batch, n_head, tokens, hs), "-> out:", tuple(out.shape),
      " weights:", tuple(w.shape))

Now the test that every causal attention must pass: perturb the future, measure
the change at position 0. It must be zero to floating-point tolerance.

In [ ]:
k2 = k.clone()
v2 = v.clone()
k2[:, :, 1:] += 3.0
v2[:, :, 1:] += 3.0
out2, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
leak = (out[:, :, 0] - out2[:, :, 0]).abs().max().item()
print("max change at position 0 from editing the future:", leak)
assert tuple(out.shape) == (batch, n_head, tokens, hs)
assert tuple(w.shape) == (batch, n_head, tokens, tokens)
assert leak < 1e-6